In [ ]:
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer

seed = 42
trained_weights = "mistralai/Mistral-7B-Instruct-v0.2"

## Load

In [2]:
ds = load_dataset("virattt/financial-qa-10K")

tokenizer = AutoTokenizer.from_pretrained(trained_weights)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

df = ds["train"].to_pandas()
df.head()

,question,answer,context,ticker,filing
0,What area did NVIDIA initially focus on before...,NVIDIA initially focused on PC graphics.,"Since our original focus on PC graphics, we ha...",NVDA,2023_10K
1,What are some of the recent applications of GP...,Recent applications of GPU-powered deep learni...,Some of the most recent applications of GPU-po...,NVDA,2023_10K
2,What significant invention did NVIDIA create i...,NVIDIA invented the GPU in 1999.,Our invention of the GPU in 1999 defined moder...,NVDA,2023_10K
3,How does NVIDIA's platform strategy contribute...,NVIDIA's platform strategy brings together har...,"NVIDIA has a platform strategy, bringing toget...",NVDA,2023_10K
4,What does NVIDIA's CUDA programming model enable?,NVIDIA's CUDA programming model opened the par...,With our introduction of the CUDA programming ...,NVDA,2023_10K


## Dataset Sizes

In [3]:
n_70ctx   = len(df)
n_30noctx = int(n_70ctx * 0.3 / 0.7)
dataset_size = n_70ctx + n_30noctx

split_proportion = {"train": 0.8, "val": 0.1, "test": 0.1}
train_size = int(dataset_size * split_proportion["train"])
val_size   = int(dataset_size * split_proportion["val"])
test_size  = dataset_size - train_size - val_size

## Assign Prompt Types

In [4]:
# apply prompt types in the question_only first
question_only = df.sample(n=n_30noctx, random_state=seed).copy()
question_only["prompt_type"] = "question_only"

df["prompt_type"] = "context_grounded"

new_df = pd.concat([df, question_only]).reset_index(drop=True)
new_df.head()

,question,answer,context,ticker,filing,prompt_type
0,What area did NVIDIA initially focus on before...,NVIDIA initially focused on PC graphics.,"Since our original focus on PC graphics, we ha...",NVDA,2023_10K,context_grounded
1,What are some of the recent applications of GP...,Recent applications of GPU-powered deep learni...,Some of the most recent applications of GPU-po...,NVDA,2023_10K,context_grounded
2,What significant invention did NVIDIA create i...,NVIDIA invented the GPU in 1999.,Our invention of the GPU in 1999 defined moder...,NVDA,2023_10K,context_grounded
3,How does NVIDIA's platform strategy contribute...,NVIDIA's platform strategy brings together har...,"NVIDIA has a platform strategy, bringing toget...",NVDA,2023_10K,context_grounded
4,What does NVIDIA's CUDA programming model enable?,NVIDIA's CUDA programming model opened the par...,With our introduction of the CUDA programming ...,NVDA,2023_10K,context_grounded


## Shuffle & Split

In [5]:
df_shuffled = new_df.sample(frac=1, random_state=seed).reset_index(drop=True)
display(df_shuffled.head())

train_df = df_shuffled.iloc[:train_size]
val_df   = df_shuffled.iloc[train_size:train_size + val_size]
test_df  = df_shuffled.iloc[train_size + val_size:]

# Save datasets
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

,question,answer,context,ticker,filing,prompt_type
0,What pages in IBM's 2023 Annual Report to Stoc...,Pages 44 through 121,Financial Statements and Supplementary Data ar...,IBM,2023_10K,context_grounded
1,What impact did the adoption of ASU 2021-08 ha...,The adoption of ASU 2021-08 did not have an im...,Recently Adopted Accounting Pronouncements In ...,ENPH,2023_10K,context_grounded
2,What distinguishes the components of Google Cl...,Google Cloud revenues consist of consumption-b...,Google Cloud revenues are comprised of Google ...,GOOGL,2023_10K,context_grounded
3,What factors led to the 7% decrease in U.S. In...,The decrease was primarily due to the negative...,U.S. Information Solutions revenue decreased 7...,EFX,2023_10K,context_grounded
4,What was the change in operating cash flow for...,decreased by 18 percent,The operating cash flow for Electronic Arts in...,EA,2023_10K,context_grounded


Train: 8000 | Val: 1000 | Test: 1000


## Evaluate prompt_type per split

In [7]:
train_df["prompt_type"].value_counts(normalize=True)

prompt_type
context_grounded    0.7025
question_only       0.2975
Name: proportion, dtype: float64

In [8]:
val_df["prompt_type"].value_counts(normalize=True)

prompt_type
context_grounded    0.677
question_only       0.323
Name: proportion, dtype: float64

In [9]:
test_df["prompt_type"].value_counts(normalize=True)

prompt_type
context_grounded    0.703
question_only       0.297
Name: proportion, dtype: float64

## Save

In [14]:
from pathlib import Path

# for .py in scripts
# PROJECT_ROOT  = Path(__file__).resolve().parent.parent

# for notebooks folder
PROJECT_ROOT = Path.cwd().parent
project_root = str(PROJECT_ROOT)

project_root

'/var/home/anne/Documents/_Ironhack/alpha-crunch'

In [16]:
output_dir = PROJECT_ROOT / "data" / "fiqa"

if not output_dir.is_dir():
    output_dir.mkdir()
    

In [19]:
train_csv = "train_df.csv"
val_csv = "val_df.csv"
test_csv = "test_df.csv"

train_df.to_csv(str(output_dir / train_csv), index=False)
val_df.to_csv(str(output_dir / val_csv), index=False)
test_df.to_csv(str(output_dir / test_csv), index=False)

## Loading from data folder

In [21]:
train_df_loaded = pd.read_csv(str(output_dir / train_csv))
train_df_loaded.head()

,question,answer,context,ticker,filing,prompt_type
0,What pages in IBM's 2023 Annual Report to Stoc...,Pages 44 through 121,Financial Statements and Supplementary Data ar...,IBM,2023_10K,context_grounded
1,What impact did the adoption of ASU 2021-08 ha...,The adoption of ASU 2021-08 did not have an im...,Recently Adopted Accounting Pronouncements In ...,ENPH,2023_10K,context_grounded
2,What distinguishes the components of Google Cl...,Google Cloud revenues consist of consumption-b...,Google Cloud revenues are comprised of Google ...,GOOGL,2023_10K,context_grounded
3,What factors led to the 7% decrease in U.S. In...,The decrease was primarily due to the negative...,U.S. Information Solutions revenue decreased 7...,EFX,2023_10K,context_grounded
4,What was the change in operating cash flow for...,decreased by 18 percent,The operating cash flow for Electronic Arts in...,EA,2023_10K,context_grounded


## Preprocessing for training

In [26]:
def format_row(row, tokenizer, add_answer=True):

    format = "Use the following context to answer the question.\n\nContext: {context}\n\nQuestion: {question}"

    try:
    
        if row["prompt_type"] == "context_grounded":
            content = format.format(context=row["context"], question=row["question"])
        elif row["prompt_type"] == "question_only":
            content = row["question"]
        else:
            raise ValueError(f"Invalid prompt_type: {row['prompt_type']}")


        chat = [{"role": "user", "content": content}]

        if add_answer: 
            chat.append({"role": "assistant", "content": row["answer"]})

            return tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=False)
    
        else:
            return tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)

    except Exception as e:
        print(f"[format_row error] {e}")
        return None


In [23]:
def build_hf_dataset(df, tokenizer):

    formatted = df.apply(lambda x: format_row(x, tokenizer), axis=1).tolist()
    formatted = [x for x in formatted if x is not None]  # filter failed rows

    return Dataset.from_dict({"chat": formatted})


train_dataset = build_hf_dataset(train_df, tokenizer)
val_dataset   = build_hf_dataset(val_df,   tokenizer)
# test_dataset  = build_hf_dataset(test_df,  tokenizer) # if I want to validate with the same pipeline of the training

dataset_splits = {
    "train": train_dataset,
    "val":   val_dataset,
    # "test":  test_dataset,
}


In [24]:
dataset_splits

{'train': Dataset({
     features: ['chat'],
     num_rows: 8000
 }),
 'val': Dataset({
     features: ['chat'],
     num_rows: 1000
 })}

## Preprocessing for inference

In [25]:
test_df_loaded = pd.read_csv(str(output_dir / test_csv))
test_df_loaded.head()

,question,answer,context,ticker,filing,prompt_type
0,What are the three levels of inputs defined in...,The three levels of inputs defined in the fair...,The fair value hierarchy described includes th...,INTU,2023_10K,context_grounded
1,What factors influenced the decrease in net bo...,The decrease in net bookings for fiscal year 2...,Net bookings decreased in fiscal year 2023 by ...,EA,2023_10K,question_only
2,What speeds does the company's domestic broadb...,The company offers downstream speeds up to 2 g...,In connection with a multiyear network transfo...,CMCSA,2023_10K,question_only
3,What are the main categories of revenue listed...,The main categories of revenue listed are Prod...,Consolidated Statements of Operations include ...,CVS,2023_10K,question_only
4,What was the operating revenue for the twelve ...,"$5,265.2 million",Consolidated operating revenue for the twelve ...,EFX,2023_10K,question_only


In [27]:
inf_inputs = test_df_loaded.apply(lambda x: format_row(x, tokenizer, add_answer=False), axis=1).tolist()
inf_inputs = [x for x in inf_inputs if x is not None]  # filter failed rows


In [29]:
inf_inputs[0]

'<s> [INST] Use the following context to answer the question.\n\nContext: The fair value hierarchy described includes three levels of inputs: Level 1 uses unadjusted quoted prices in active markets for identical assets or liabilities, Level 2 uses inputs that are directly or indirectly observable, and Level 3 involves unobservable inputs significant to the fair value determination supported by minimal market activity.\n\nQuestion: What are the three levels of inputs defined in the fair value hierarchy? [/INST]'